In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score


In [ ]:
data_path = "../../data/Student_performance_data _.csv"
df = pd.read_csv(data_path)
print(df.shape)


In [ ]:
target_col = "GradeClass"
true_labels = df[target_col].values
X_cluster = df.drop(columns=["StudentID", target_col])


In [ ]:
scaler = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
X_scaled = scaler.fit_transform(X_cluster)


In [ ]:
configs = [
    {"n_clusters": 5, "linkage": "ward"},
    {"n_clusters": 5, "linkage": "average"},
    {"n_clusters": 5, "linkage": "complete"},
    {"n_clusters": 4, "linkage": "ward"},
    {"n_clusters": 6, "linkage": "ward"},
]

best_cfg = None
best_sil = -1
best_labels = None

for cfg in configs:
    model = AgglomerativeClustering(**cfg)
    labels = model.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    ari = adjusted_rand_score(true_labels, labels)
    nmi = normalized_mutual_info_score(true_labels, labels)
    print(f"{cfg} | silhouette={sil:.4f} | ARI={ari:.4f} | NMI={nmi:.4f}")
    if sil > best_sil:
        best_sil = sil
        best_cfg = cfg
        best_labels = labels

print("\nBest config:", best_cfg)


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

os.makedirs("visualizations", exist_ok=True)
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=best_labels, cmap="tab10", alpha=0.6, s=20)
plt.colorbar(scatter, label="Cluster")
plt.title(f"Hierarchical Clustering - PCA 2D ({best_cfg})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.savefig("visualizations/hierarchical_clusters_pca.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
results = pd.DataFrame({
    "Algorithm": ["Hierarchical"],
    "Best_Config": [str(best_cfg)],
    "Silhouette": [best_sil],
    "ARI": [adjusted_rand_score(true_labels, best_labels)],
    "NMI": [normalized_mutual_info_score(true_labels, best_labels)]
})
os.makedirs("results", exist_ok=True)
results.to_csv("results/hierarchical_results.csv", index=False)
print(results)
